# FlakyGuard AI: Exploratory Data Analysis (EDA)
**Study**: "Test Flakiness Across Programming Languages" (Barbosa et al., IEEE TSE 2023)
**Objective**: Comprehensive analysis of GitHub issues related to test flakiness across C, Go, Java, JavaScript, and Python.

This notebook executes the dataset audit, distribution analyses, cross-language comparisons, text length investigations, and vocabulary examinations.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

PROCESSED_DIR = '../data/processed'
RAW_DIR = '../data/raw'
FIGURES_DIR = '../artifacts/figures'

binary_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'flaky_binary_dataset.csv'))
rc_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'root_cause_dataset.csv'))
full_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'full_cleaned_dataset.csv'))

print(f"Full Cleaned Dataset: {full_df.shape}")
print(f"Task A Binary Dataset: {binary_df.shape}")
print(f"Task B Root-Cause Dataset: {rc_df.shape}")


## 1. Dataset Overview & Schema Inspection
Inspecting the high-level distributions, unique projects, and column characteristics.


In [ ]:
full_df.info()
print("\nDistinct repositories:", full_df['repository'].nunique())
print("Languages represented:", full_df['language'].value_counts().to_dict())


## 2. Flakiness Status Distribution (T, F, ND)
Understanding ground-truth class proportions and the extent of class imbalance between confirmed flaky tests (`T`) and false positives (`F`).


In [ ]:
status_counts = full_df['flakiness_status'].value_counts(dropna=False)
display(pd.DataFrame({'Count': status_counts, 'Percentage (%)': (status_counts / len(full_df) * 100).round(2)}))

plt.figure(figsize=(7, 4.5))
ax = sns.barplot(x=status_counts.index, y=status_counts.values, palette=['#4361EE', '#E63946', '#F77F00', '#6C757D'])
plt.title("Ground-Truth Flakiness Distribution (TSE22 Cleaned)")
plt.xlabel("Status")
plt.ylabel("Number of Issues")
for p in ax.patches:
    h = p.get_height()
    ax.annotate(f"{int(h)} ({h/len(full_df)*100:.1f}%)", (p.get_x() + p.get_width() / 2., h / 2),
                ha='center', va='center', color='white', weight='bold')
plt.tight_layout()
plt.show()


## 3. Cross-Language Flakiness Analysis
Evaluating whether the proportion of true flaky issues vs false positives varies across C, Go, Java, JavaScript, and Python.


In [ ]:
lang_crosstab = pd.crosstab(full_df['language'], full_df['flakiness_status'], normalize='index') * 100
display(lang_crosstab.round(1))

plt.figure(figsize=(9, 5))
pd.crosstab(full_df['language'], full_df['flakiness_status'])[['T', 'F', 'ND']].plot(
    kind='bar', stacked=True, color=['#4361EE', '#E63946', '#F77F00'], figsize=(9, 5)
)
plt.title("Flakiness Breakdown by Programming Language")
plt.xlabel("Language")
plt.ylabel("Issues Count")
plt.legend(["Flaky (T)", "False Positive (F)", "Not Determined (ND)"])
plt.tight_layout()
plt.show()


## 4. Root-Cause Category Distribution
Inspecting the 13 confirmed root-cause categories categorized by researchers.


In [ ]:
rc_counts = rc_df['root_cause_category'].value_counts()
display(pd.DataFrame({'Count': rc_counts, 'Share (%)': (rc_counts / len(rc_df) * 100).round(2)}))

plt.figure(figsize=(10, 5.5))
rc_counts.plot(kind='barh', color=sns.color_palette('mako', len(rc_counts)))
plt.title("Confirmed Root-Cause Categories in TSE22 (N=589)")
plt.xlabel("Count")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 5. Issue Text Length Analysis
Comparing word counts between true flaky issues and false positives to detect potential length artifacts.


In [ ]:
plt.figure(figsize=(8, 4.5))
sns.kdeplot(data=binary_df[binary_df['label'] == 1]['text_length'], label='Flaky (T)', color='#4361EE', fill=True, alpha=0.3)
sns.kdeplot(data=binary_df[binary_df['label'] == 0]['text_length'], label='False Positive (F)', color='#E63946', fill=True, alpha=0.3)
plt.title("Word Length Distribution: Flaky vs False Positive Issues")
plt.xlabel("Word Count")
plt.xlim(0, 1000)
plt.legend()
plt.tight_layout()
plt.show()


## 6. Correlation Analysis for Available Metadata
Analyzing correlation between numerical metrics (days to close M1, comments count M2, year, text length) and flakiness status.


In [ ]:
corr_data = full_df[['flakiness_status', 'days_to_close', 'comments_count', 'year']].copy()
corr_data['is_flaky'] = corr_data['flakiness_status'].map({'T': 1, 'F': 0})
corr_data['days_to_close'] = pd.to_numeric(corr_data['days_to_close'], errors='coerce')
corr_data['comments_count'] = pd.to_numeric(corr_data['comments_count'], errors='coerce')
corr_data['year'] = pd.to_numeric(corr_data['year'], errors='coerce')

plt.figure(figsize=(6, 4.5))
sns.heatmap(corr_data[['is_flaky', 'days_to_close', 'comments_count', 'year']].corr(),
            annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Metadata Correlation Matrix")
plt.tight_layout()
plt.show()
